In [32]:
import pandas as pd
import numpy as np
import math

In [33]:
file="C:\\Users\\amtha\\Videos\\PF Assistant\\dataSet.csv"

In [34]:
df = pd.read_csv(file)
df

,Device,Type,Power(W),PF
0,Ceiling_Fan,Inductive,75,0.70
1,LED_Bulb,Electronic,12,0.90
2,Refrigerator,Inductive,150,0.82
3,Electric_Heater,Resistive,1500,1.00
4,Air_Conditioner,Inductive,1200,0.85
5,Laptop_Charger,Electronic,65,0.95
6,Washing_Machine,Inductive,500,0.75
7,Incandescent_Lamp,Resistive,60,1.00
8,Microwave_Oven,Inductive,1100,0.88
9,Desktop_PC,Electronic,300,0.92


In [35]:
df["phase_angle_degrees"] = round(np.degrees(np.arccos(df["PF"])),3)
df["Reactive Power(VAR)"] = round(df["Power(W)"]*np.tan(np.radians(df["phase_angle_degrees"])),3)

df

,Device,Type,Power(W),PF,phase_angle_degrees,Reactive Power(VAR)
0,Ceiling_Fan,Inductive,75,0.70,45.573,76.515
1,LED_Bulb,Electronic,12,0.90,25.842,5.812
2,Refrigerator,Inductive,150,0.82,34.915,104.700
3,Electric_Heater,Resistive,1500,1.00,0.000,0.000
4,Air_Conditioner,Inductive,1200,0.85,31.788,743.684
5,Laptop_Charger,Electronic,65,0.95,18.195,21.365
6,Washing_Machine,Inductive,500,0.75,41.410,440.964
7,Incandescent_Lamp,Resistive,60,1.00,0.000,0.000
8,Microwave_Oven,Inductive,1100,0.88,28.358,593.726
9,Desktop_PC,Electronic,300,0.92,23.074,127.800


In [36]:
P_total = df["Power(W)"].sum()
Q_total = df["Reactive Power(VAR)"].sum()

S_total = math.sqrt((P_total)**2 + (Q_total)**2)

PF_total = P_total / S_total

print(
    f"P_total: {P_total:.2f} W\n",
    f"Q_total: {Q_total:.2f} VAR\n",
    f"S_total: {S_total:.2f} VA\n",
    f"PF_total: {PF_total:.2f}\n"
)

#expected PF = 0.95
#Q_c = P*(tan(theta1)-tan(theta2))
#Q_c is required reactive power

Q_c = P_total*(np.tan(np.arccos(PF_total))-np.tan(np.arccos(0.95)))

print(f"Q_c: {Q_c:.2f} VAR")

P_total: 5812.00 W
 Q_total: 2857.77 VAR
 S_total: 6476.59 VA
 PF_total: 0.90

Q_c: 947.46 VAR


In [37]:
df["Q_contribution_pct(%)"] = round((df["Reactive Power(VAR)"] / Q_total) * 100,2)
df[["Device", "Reactive Power(VAR)", "Q_contribution_pct(%)"]]

,Device,Reactive Power(VAR),Q_contribution_pct(%)
0,Ceiling_Fan,76.515,2.68
1,LED_Bulb,5.812,0.20
2,Refrigerator,104.700,3.66
3,Electric_Heater,0.000,0.00
4,Air_Conditioner,743.684,26.02
5,Laptop_Charger,21.365,0.75
6,Washing_Machine,440.964,15.43
7,Incandescent_Lamp,0.000,0.00
8,Microwave_Oven,593.726,20.78
9,Desktop_PC,127.800,4.47


In [38]:
usage_prob = {
    "Ceiling_Fan": 0.6,
    "LED_Bulb": 0.8,
    "Refrigerator": 0.9, 
    "Electric_Heater": 0.2,
    "Air_Conditioner": 0.5,
    "Laptop_Charger": 0.4,
    "Washing_Machine": 0.1,
    "Incandescent_Lamp": 0.3,
    "Microwave_Oven": 0.1,
    "Desktop_PC": 0.3,
    "Water_Pump": 0.05,
    "Television": 0.4
}

In [39]:
import random

def generate_scenario(df):
    df_copy = df.copy()
    
    df_copy["ON"] = df_copy["Device"].apply(
        lambda x: 1 if random.random() < usage_prob[x] else 0
    )
    
    return df_copy

In [40]:
def compute_totals(df):
    P_total = round((df["Power(W)"] * df["ON"]).sum(),3)
    Q_total = round((df["Reactive Power(VAR)"] * df["ON"]).sum(),3)
    
    if P_total == 0:
        return 0, 0, 0
    
    S_total = round(math.sqrt(P_total**2 + Q_total**2),3)
    PF_total = round(P_total / S_total,3)
    
    return P_total, Q_total, PF_total

In [41]:
scenarios = []

for i in range(500):
    scenario_df = generate_scenario(df)
    
    P, Q, PF = compute_totals(scenario_df)
    
    # Create a dictionary for ON/OFF states
    scenario_dict = {}
    
    for _, row in scenario_df.iterrows():
        scenario_dict[row["Device"]] = row["ON"]
    
    # Add totals
    scenario_dict["P_total"] = P
    scenario_dict["Q_total"] = Q
    scenario_dict["PF_total"] = PF
    
    scenarios.append(scenario_dict)

    target_pf = 0.95

    Qc = P * (np.tan(np.arccos(PF)) - np.tan(np.arccos(target_pf)))
    scenario_dict["Q_c"] = Qc

    scenario_dict["num_devices_on"] = scenario_df["ON"].sum()

In [42]:
scenarios

[{'Ceiling_Fan': 1,
  'LED_Bulb': 1,
  'Refrigerator': 1,
  'Electric_Heater': 1,
  'Air_Conditioner': 0,
  'Laptop_Charger': 0,
  'Washing_Machine': 0,
  'Incandescent_Lamp': 0,
  'Microwave_Oven': 0,
  'Desktop_PC': 0,
  'Water_Pump': 0,
  'Television': 0,
  'P_total': np.int64(1737),
  'Q_total': np.float64(187.027),
  'PF_total': np.float64(0.994),
  'Q_c': np.float64(-379.7842666844062),
  'num_devices_on': np.int64(4)},
 {'Ceiling_Fan': 0,
  'LED_Bulb': 1,
  'Refrigerator': 1,
  'Electric_Heater': 0,
  'Air_Conditioner': 1,
  'Laptop_Charger': 1,
  'Washing_Machine': 0,
  'Incandescent_Lamp': 1,
  'Microwave_Oven': 0,
  'Desktop_PC': 0,
  'Water_Pump': 0,
  'Television': 0,
  'P_total': np.int64(1487),
  'Q_total': np.float64(875.561),
  'PF_total': np.float64(0.862),
  'Q_c': np.float64(385.692914259455),
  'num_devices_on': np.int64(5)},
 {'Ceiling_Fan': 0,
  'LED_Bulb': 1,
  'Refrigerator': 1,
  'Electric_Heater': 0,
  'Air_Conditioner': 0,
  'Laptop_Charger': 0,
  'Washing_Ma

In [43]:
print(generate_scenario(df))

               Device        Type  Power(W)    PF  phase_angle_degrees  \
0         Ceiling_Fan   Inductive        75  0.70               45.573   
1            LED_Bulb  Electronic        12  0.90               25.842   
2        Refrigerator   Inductive       150  0.82               34.915   
3     Electric_Heater   Resistive      1500  1.00                0.000   
4     Air_Conditioner   Inductive      1200  0.85               31.788   
5      Laptop_Charger  Electronic        65  0.95               18.195   
6     Washing_Machine   Inductive       500  0.75               41.410   
7   Incandescent_Lamp   Resistive        60  1.00                0.000   
8      Microwave_Oven   Inductive      1100  0.88               28.358   
9          Desktop_PC  Electronic       300  0.92               23.074   
10         Water_Pump   Inductive       750  0.72               43.946   
11         Television  Electronic       100  0.98               11.478   

    Reactive Power(VAR)  Q_contributi

In [44]:
dataset = pd.DataFrame(scenarios)

In [46]:
dataset.sample(30)

,Ceiling_Fan,LED_Bulb,Refrigerator,Electric_Heater,Air_Conditioner,Laptop_Charger,Washing_Machine,Incandescent_Lamp,Microwave_Oven,Desktop_PC,Water_Pump,Television,P_total,Q_total,PF_total,Q_c,num_devices_on
288,0,0,1,0,1,0,0,1,0,0,0,1,1510,868.689,0.867,371.559305,4
50,0,1,1,0,1,1,0,1,0,0,0,1,1587,895.866,0.871,373.517411,6
13,1,1,1,1,0,1,0,0,0,1,0,1,2202,356.497,0.987,-365.195025,7
126,1,0,1,0,0,0,0,0,0,0,0,0,225,181.215,0.779,107.150547,2
391,1,1,1,0,0,0,1,1,0,0,0,1,897,648.296,0.810,354.587147,6
72,1,0,1,0,1,0,0,0,0,0,0,0,1425,924.899,0.839,455.805449,3
486,0,1,1,0,1,1,0,0,1,0,0,0,2527,1469.287,0.864,642.012253,5
364,1,1,1,0,1,0,0,0,0,0,0,1,1537,951.016,0.850,447.359578,5
281,1,1,0,0,1,0,0,0,0,0,0,0,1287,826.011,0.842,401.576876,3
34,0,0,1,0,1,0,0,0,0,0,0,1,1450,868.689,0.858,391.464044,3
